In [2]:
from typing import Mapping, Sequence
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import pulp

In [3]:
# Constants

INPUT_BASE_PATH = Path("../data")
FORECASTS_PATH = Path("../results/ets/submissions")


# ==================
# FORECAST CONSTANTS 
# ==================
MAX_TRAINING_TIMESTAMP = 1913
FORECAST_HORIZON = 7
N_FORECAST_TIMESTAMPS = 4

# Timestamps at which we will generate forecasts. We will assume that we have already
# observed the ground truth value of each product series at the forecast timestamp and
# we produce forcasts for the following FORECAST_HORIZON timestamps.
FORECAST_TIMESTAMPS = [MAX_TRAINING_TIMESTAMP + i * FORECAST_HORIZON for i in range(N_FORECAST_TIMESTAMPS)]
FORECASTED_TIMESTAMPS = [list(range(t + 1, t + 1 + FORECAST_HORIZON)) for t in FORECAST_TIMESTAMPS]

In [4]:
# Load data

CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")

In [5]:
#  

SALES_DF = (
    SALES_TRAIN_EVALUATION
    .with_columns(
        id=pl.col("id").str.strip_suffix("_evaluation")
    )
    .unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales"
    ).with_columns(
        d_index=pl.col("d").str.extract(r"^d_([0-9]+)").cast(pl.Int64)
    )
)

SALES_TRAIN_DF = SALES_DF.filter(pl.col("d_index") <= MAX_TRAINING_TIMESTAMP)
SALES_VAL_DF = SALES_DF.filter(pl.col("d_index") > MAX_TRAINING_TIMESTAMP)

assert min(FORECAST_TIMESTAMPS) == SALES_TRAIN_DF["d_index"].max()
assert all(t in SALES_VAL_DF["d_index"].unique().to_list() for t in sum(FORECASTED_TIMESTAMPS, []))

In [16]:
# Baseline forecasts

# 1. Weekly average
grouper_df = (
    SALES_TRAIN_DF
    .filter(pl.col("id").is_in(["HOBBIES_1_001_CA_1", "HOBBIES_1_002_CA_1"]))
    .sort(pl.col("id"), pl.col("d_index"))
    .rolling(pl.col("d_index"), period="3i", group_by=pl.col("id"))
    .agg(
        indices_in_group=pl.col("d_index"),
        avg_sales=pl.col("sales").mean()
    )
)

# 2. Perfect information
